In [1]:
import os
os.chdir("../../..")

In [2]:
import torch
from models.DeepCNN import DeepCNN
from models.RamanFormer import RamanFormer
from models.RamanNet import RamanNet
from models.SANet import ScaleAdaptiveNet
from models.transformer import ViT
from torch.utils.data import Dataset,DataLoader
import pickle
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

In [3]:
class Bacteria_Dataset(Dataset):
    def __init__(self,X_path,y_path,num_classes=30):
        """
        X_path is a string containing the path to the spectra
        y_path is a string containing the path to the labels for the spectra
        num_classes is an integer corresponding to an 8 class or 30 class problem 
        """
        super().__init__()
        self.X = np.load(X_path)
        self.y = np.load(y_path)
        self.num_classes = num_classes
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self,index):
        data = torch.Tensor(self.X[index]).unsqueeze(0) #of shape (1,1000)
        data = (data-data.min())/(data.max()-data.min())
        label = self.y[index]
        if self.num_classes==8:
            label = ATCC_GROUPINGS[label]
        return data,int(label)

In [4]:
def test_f1(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [5]:
def test_f1_RamanNet(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred,_ = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [6]:
def test_f1_wrapper(model1,model2,model3,model4,model5,device,test_dataloader,RamanNet=False):

    if not RamanNet:
        all_acc1, _, _, all_f11 = test_f1(model1,device,test_dataloader)
        all_acc2, _, _, all_f12 = test_f1(model2,device,test_dataloader)
        all_acc3, _, _, all_f13 = test_f1(model3,device,test_dataloader)
        all_acc4, _, _, all_f14 = test_f1(model4,device,test_dataloader)
        all_acc5, _, _, all_f15 = test_f1(model5,device,test_dataloader)

    else:
        all_acc1, _, _, all_f11 = test_f1_RamanNet(model1,device,test_dataloader)
        all_acc2, _, _, all_f12 = test_f1_RamanNet(model2,device,test_dataloader)
        all_acc3, _, _, all_f13 = test_f1_RamanNet(model3,device,test_dataloader)
        all_acc4, _, _, all_f14 = test_f1_RamanNet(model4,device,test_dataloader)
        all_acc5, _, _, all_f15 = test_f1_RamanNet(model5,device,test_dataloader)

    all_acc = np.array([all_acc1,all_acc2,all_acc3,all_acc4,all_acc5])
    all_f1 = np.array([all_f11,all_f12,all_f13,all_f14,all_f15])

    print(f"{round(all_acc.mean(),2)}\\% (\u00B1 {round(all_acc.std(),2)})& {round(all_f1.mean(),4)} (\u00B1 {round(all_f1.std(),4)}) \\\\")

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [8]:
test_set = Bacteria_Dataset("datasets/Bacteria_ID/X_test.npy","datasets/Bacteria_ID/y_test.npy",30)
test_loader = DataLoader(test_set, batch_size=32, num_workers=8,shuffle=False)
print(f"The number of elements in the test set is {len(test_loader.dataset)}")

The number of elements in the test set is 3000


In [11]:
deepcnn1 = DeepCNN(num_classes=30,sp_size=1000).to(device)
deepcnn1.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_deepcnn_fine_0_15_90.5_.pt",weights_only=True))

deepcnn2 = DeepCNN(num_classes=30,sp_size=1000).to(device)
deepcnn2.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_deepcnn_fine_1_9_93.33_.pt",weights_only=True))

deepcnn3 = DeepCNN(num_classes=30,sp_size=1000).to(device)
deepcnn3.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_deepcnn_fine_2_11_93.83_.pt",weights_only=True))

deepcnn4 = DeepCNN(num_classes=30,sp_size=1000).to(device)
deepcnn4.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_deepcnn_fine_3_17_91.83_.pt",weights_only=True))

deepcnn5 = DeepCNN(num_classes=30,sp_size=1000).to(device)
deepcnn5.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_deepcnn_fine_4_15_92.5_.pt",weights_only=True))

RamanFormer1 = RamanFormer(num_classes=30,sp_size=1000,patch_size=125).to(device)
RamanFormer1.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanFormer_fine_0_37_92.0_.pt",weights_only=True))

RamanFormer2 = RamanFormer(num_classes=30,sp_size=1000,patch_size=125).to(device)
RamanFormer2.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanFormer_fine_1_34_93.0_.pt",weights_only=True))

RamanFormer3 = RamanFormer(num_classes=30,sp_size=1000,patch_size=125).to(device)
RamanFormer3.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanFormer_fine_2_39_93.0_.pt",weights_only=True))

RamanFormer4 = RamanFormer(num_classes=30,sp_size=1000,patch_size=125).to(device)
RamanFormer4.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanFormer_fine_3_37_94.5_.pt",weights_only=True))

RamanFormer5 = RamanFormer(num_classes=30,sp_size=1000,patch_size=125).to(device)
RamanFormer5.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanFormer_fine_4_36_92.5_.pt",weights_only=True))

RamanNet1 = RamanNet(num_classes=30,sp_size=1000).to(device)
RamanNet1.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanNet_fine_0_36_87.67_.pt",weights_only=True))

RamanNet2 = RamanNet(num_classes=30,sp_size=1000).to(device)
RamanNet2.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanNet_fine_1_37_91.83_.pt",weights_only=True))

RamanNet3 = RamanNet(num_classes=30,sp_size=1000).to(device)
RamanNet3.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanNet_fine_2_36_91.5_.pt",weights_only=True))

RamanNet4 = RamanNet(num_classes=30,sp_size=1000).to(device)
RamanNet4.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanNet_fine_3_35_91.17_.pt",weights_only=True))

RamanNet5 = RamanNet(num_classes=30,sp_size=1000).to(device)
RamanNet5.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_RamanNet_fine_4_38_89.67_.pt",weights_only=True))

SANet1 = ScaleAdaptiveNet(num_classes=30).to(device)
SANet1.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_SANet_fine_0_36_90.83_.pt",weights_only=True))

SANet2 = ScaleAdaptiveNet(num_classes=30).to(device)
SANet2.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_SANet_fine_1_27_91.5_.pt",weights_only=True))

SANet3 = ScaleAdaptiveNet(num_classes=30).to(device)
SANet3.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_SANet_fine_2_38_92.0_.pt",weights_only=True))

SANet4 = ScaleAdaptiveNet(num_classes=30).to(device)
SANet4.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_SANet_fine_3_39_92.67_.pt",weights_only=True))

SANet5 = ScaleAdaptiveNet(num_classes=30).to(device)
SANet5.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_SANet_fine_4_39_90.0_.pt",weights_only=True))

transformer1 = ViT(patch_size=125,p=0.1,num_classes=30).to(device)
transformer1.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_transformer_fine_0_23_91.83_.pt",weights_only=True))

transformer2 = ViT(patch_size=125,p=0.1,num_classes=30).to(device)
transformer2.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_transformer_fine_1_29_94.83_.pt",weights_only=True))

transformer3 = ViT(patch_size=125,p=0.1,num_classes=30).to(device)
transformer3.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_transformer_fine_2_18_93.17_.pt",weights_only=True))

transformer4 = ViT(patch_size=125,p=0.1,num_classes=30).to(device)
transformer4.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_transformer_fine_3_30_92.67_.pt",weights_only=True))

transformer5 = ViT(patch_size=125,p=0.1,num_classes=30).to(device)
transformer5.load_state_dict(torch.load("results/final_multi_run/trained_models/Bacteria_ID_thirty_transformer_fine_4_38_91.17_.pt",weights_only=True))

<All keys matched successfully>

In [12]:
test_f1_wrapper(deepcnn1,deepcnn2,deepcnn3,deepcnn4,deepcnn5,device,test_loader,RamanNet=False)

100%|██████████| 94/94 [00:00<00:00, 205.66it/s]

85.89\% (± 0.12)& 0.8565 (± 0.0009) \\


In [13]:
test_f1_wrapper(RamanFormer1,RamanFormer2,RamanFormer3,RamanFormer4,RamanFormer5,device,test_loader,RamanNet=False)

100%|██████████| 94/94 [00:00<00:00, 195.90it/s]

84.63\% (± 0.58)& 0.8424 (± 0.0063) \\


In [14]:
test_f1_wrapper(RamanNet1,RamanNet2,RamanNet3,RamanNet4,RamanNet5,device,test_loader,RamanNet=True)

100%|██████████| 94/94 [00:00<00:00, 118.39it/s]

84.06\% (± 0.76)& 0.8392 (± 0.0082) \\


In [15]:
test_f1_wrapper(SANet1,SANet2,SANet3,SANet4,SANet5,device,test_loader,RamanNet=False)

100%|██████████| 94/94 [00:00<00:00, 119.04it/s]

85.62\% (± 0.6)& 0.8547 (± 0.0063) \\


In [16]:
test_f1_wrapper(transformer1,transformer2,transformer3,transformer4,transformer5,device,test_loader,RamanNet=False)

100%|██████████| 94/94 [00:00<00:00, 108.62it/s]

83.27\% (± 0.62)& 0.827 (± 0.0069) \\
